In [ ]:
import sys
sys.path.append("..")

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from PIL import Image

import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

from model.upscaler import SuperResNet
from model.dataset import SuperResDataset
from model.lit_upscaler import ImageLoggerCallback, LitSuperResNet
import matplotlib.pyplot as plt

from skimage.io import imread
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
TEST_PATH = os.getenv("TEST_DATA_PATH")
_3RDPARTY_PATH = os.getenv("_3RDPARTY_PATH")

HIRES_PATCH_SIZE = 128

In [ ]:
upscaler_test = LitSuperResNet.load_from_checkpoint(
    '../model/checkpoints/v5_d3/0/upscaler-epoch=096.ckpt',
    ).to(DEVICE)

upscaler_test.eval()

In [ ]:
import torch.nn.functional as F

TEST_SEED = 43

test_dataset = SuperResDataset(
    TEST_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    seed=TEST_SEED,
)
_3rdparty_dataset = SuperResDataset(
    _3RDPARTY_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    seed=TEST_SEED,
)
test_loader = DataLoader(test_dataset,
                     batch_size=16, shuffle=False, num_workers=0)
_3rdparty_loader = DataLoader(_3rdparty_dataset,
                     batch_size=16, shuffle=False, num_workers=0)


_3rdparty_batch = next(iter(_3rdparty_loader))


for batch in test_loader:
    with torch.no_grad():
        x, y_true = upscaler_test.batch_preprocess(batch)
        y_pred = upscaler_test(x.to(DEVICE))

    for i in range(len(y_true)):
        
        fig, ax = plt.subplots(1, 4, figsize=(20,5))

        ax[0].imshow(SuperResDataset.tensor_to_pil(y_pred[i]))
        ax[0].set_title("HR Prediction")
        ax[0].axis("off") 

        ax[1].imshow(SuperResDataset.tensor_to_pil(y_true[i]))
        ax[1].set_title("HR Ground Truth (Y)")
        ax[1].axis("off")

        lowres = SuperResDataset.tensor_to_pil(x[i].cpu())
        baseline = lowres.resize(
            (x.shape[-1]*2, x.shape[-2]*2),
            resample=Image.LANCZOS
        )
        ax[2].imshow(baseline)
        ax[2].set_title("Baseline (LANCZOS)")
        ax[2].axis("off")

        _3rdparty_pic = SuperResDataset.tensor_to_pil(_3rdparty_batch[i].cpu())
        ax[3].imshow(_3rdparty_pic)
        ax[3].set_title("A")
        ax[3].axis("off")

        plt.show()
    break
